# Módulo 5 · Clase 3 — FastAPI básico

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema | |
|---|------|-|
| 1 | ¿Qué es FastAPI? | Framework moderno, rápido, con tipado |
| 2 | Instalación y primer servidor | `uvicorn`, `app = FastAPI()` |
| 3 | Path params | `/items/{id}` |
| 4 | Query params | `/items?skip=0&limit=10` |
| 5 | Pydantic — validación | Modelos de datos con tipado |
| 6 | Request body | POST / PUT con body JSON |
| 7 | Response models | Controlar qué devuelve la API |
| 8 | Manejo de errores | `HTTPException` |
| 9 | Práctica | API de tareas en memoria |

> ⚠️ FastAPI necesita un servidor ASGI (`uvicorn`) para ejecutarse. En este notebook lo lanzamos en un hilo de fondo y lo llamamos con `requests` desde la misma sesión.

In [ ]:
import subprocess, sys

for pkg in ['fastapi', 'uvicorn[standard]', 'requests', 'httpx']:
    try:
        nombre = pkg.split('[')[0]
        __import__(nombre)
        print(f"{nombre}: ya instalado")
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
        print(f"{pkg}: instalado")

---
## 1. ¿Qué es FastAPI?

FastAPI es un framework Python moderno para construir APIs:

```
VENTAJAS:
✦ Muy rápido — comparable a Node.js y Go (gracias a Starlette + uvicorn)
✦ Tipado — usa type hints de Python para validar automáticamente
✦ Documentación automática — Swagger UI y ReDoc generados solos
✦ Menos código — menos bugs, menos boilerplate
✦ Async nativo — soporta async/await desde el principio

STACK TÍPICO:
FastAPI   ← framework (define rutas, validación, serialización)
Pydantic  ← validación de datos (incluido con FastAPI)
Uvicorn   ← servidor ASGI que ejecuta la app
SQLAlchemy← ORM para base de datos (Clase 4)

COMPARACIÓN:
Flask    → minimalista, sin validación, sin async nativo
Django   → completo pero pesado, más orientado a MVC/web
FastAPI  → equilibrio: rápido, validación, async, ideal para APIs
```

---
## 2. Instalación y primer servidor

En producción ejecutarías:
```bash
uvicorn main:app --host 0.0.0.0 --port 8000 --reload
```

En el notebook, lanzamos el servidor en un hilo de fondo.

In [ ]:
from fastapi import FastAPI
import uvicorn, threading, time, requests

# ── Definir la app ───────────────────────────────────────────
app1 = FastAPI(title="Mi primera API", version="0.1.0")

@app1.get("/")
def raiz():
    return {"mensaje": "¡Hola desde FastAPI!"}

@app1.get("/salud")
def salud():
    return {"status": "ok", "version": "0.1.0"}

# ── Lanzar en hilo ───────────────────────────────────────────
config = uvicorn.Config(app1, host="127.0.0.1", port=8001, log_level="warning")
server = uvicorn.Server(config)
hilo   = threading.Thread(target=server.run, daemon=True)
hilo.start()
time.sleep(1.5)  # esperar a que arranque

# ── Probar ───────────────────────────────────────────────────
print("GET /")
print(requests.get("http://127.0.0.1:8001/").json())

print("\nGET /salud")
print(requests.get("http://127.0.0.1:8001/salud").json())

print("\n📖 Documentación interactiva disponible en:")
print("   http://127.0.0.1:8001/docs   (Swagger UI)")
print("   http://127.0.0.1:8001/redoc  (ReDoc)")

---
## 3. Path parameters

```python
@app.get("/items/{item_id}")
def get_item(item_id: int):   # FastAPI convierte y valida automáticamente
    ...

# /items/42   → item_id = 42  (int)
# /items/abc  → 422 Unprocessable Entity (no es int)
```

In [ ]:
from fastapi import FastAPI, Path
from enum import Enum

app2 = FastAPI(title="Path params demo")

# Datos simulados
PRODUCTOS = {
    1: {"id": 1, "nombre": "Teclado",    "precio": 85.0,  "categoria": "informatica"},
    2: {"id": 2, "nombre": "Ratón",      "precio": 45.0,  "categoria": "informatica"},
    3: {"id": 3, "nombre": "Smartphone", "precio": 599.0, "categoria": "electronica"},
}

class Categoria(str, Enum):
    informatica = "informatica"
    electronica = "electronica"
    hogar       = "hogar"

@app2.get("/productos")
def listar_productos():
    return list(PRODUCTOS.values())

@app2.get("/productos/{producto_id}")
def obtener_producto(
    producto_id: int = Path(description="ID del producto", ge=1)
    # ge=1 → greater or equal 1
):
    if producto_id not in PRODUCTOS:
        from fastapi import HTTPException
        raise HTTPException(status_code=404, detail=f"Producto {producto_id} no encontrado")
    return PRODUCTOS[producto_id]

@app2.get("/categorias/{categoria}")
def productos_por_categoria(categoria: Categoria):  # Enum valida los valores posibles
    return [p for p in PRODUCTOS.values() if p['categoria'] == categoria.value]


# Lanzar
cfg2 = uvicorn.Config(app2, host="127.0.0.1", port=8002, log_level="warning")
srv2 = uvicorn.Server(cfg2)
threading.Thread(target=srv2.run, daemon=True).start()
time.sleep(1.2)

BASE = "http://127.0.0.1:8002"
print("GET /productos:")
print(requests.get(f"{BASE}/productos").json())

print("\nGET /productos/1:")
print(requests.get(f"{BASE}/productos/1").json())

print("\nGET /productos/99 (404):")
r = requests.get(f"{BASE}/productos/99")
print(f"  status={r.status_code}, detail={r.json()['detail']}")

print("\nGET /productos/abc (422):")
r = requests.get(f"{BASE}/productos/abc")
print(f"  status={r.status_code}")

print("\nGET /categorias/informatica:")
print(requests.get(f"{BASE}/categorias/informatica").json())

---
## 4. Query parameters

```python
@app.get("/items")
def listar(skip: int = 0, limit: int = 10, activo: bool = True):
    ...
# GET /items?skip=20&limit=5&activo=false

# Opcional con None
def buscar(q: str | None = None):
    ...
# GET /buscar           → q = None
# GET /buscar?q=teclado → q = "teclado"
```

In [ ]:
from fastapi import FastAPI, Query
from typing import Optional

app3 = FastAPI()

@app3.get("/productos")
def listar(
    skip:       int            = Query(0,    ge=0, description="Elementos a saltar"),
    limit:      int            = Query(10,   ge=1, le=100, description="Máximo a devolver"),
    categoria:  Optional[str]  = Query(None, description="Filtrar por categoría"),
    precio_max: Optional[float]= Query(None, gt=0, description="Precio máximo"),
    q:          Optional[str]  = Query(None, min_length=2, description="Búsqueda por nombre"),
):
    items = list(PRODUCTOS.values())

    if categoria:
        items = [p for p in items if p['categoria'] == categoria]
    if precio_max is not None:
        items = [p for p in items if p['precio'] <= precio_max]
    if q:
        items = [p for p in items if q.lower() in p['nombre'].lower()]

    total = len(items)
    items = items[skip : skip + limit]
    return {"total": total, "skip": skip, "limit": limit, "items": items}


cfg3 = uvicorn.Config(app3, host="127.0.0.1", port=8003, log_level="warning")
srv3 = uvicorn.Server(cfg3)
threading.Thread(target=srv3.run, daemon=True).start()
time.sleep(1.2)

BASE3 = "http://127.0.0.1:8003"

print("GET /productos (todos):")
print(requests.get(f"{BASE3}/productos").json())

print("\nGET /productos?categoria=informatica:")
print(requests.get(f"{BASE3}/productos", params={"categoria": "informatica"}).json())

print("\nGET /productos?precio_max=100:")
print(requests.get(f"{BASE3}/productos", params={"precio_max": 100}).json())

print("\nGET /productos?q=a (422 si q < 2 chars):")
r = requests.get(f"{BASE3}/productos", params={"q": "a"})
print(f"  status={r.status_code}")

---
## 5 y 6. Pydantic y Request body

**Pydantic** define la estructura de los datos de entrada y salida:

```python
from pydantic import BaseModel, Field

class Producto(BaseModel):
    nombre:   str   = Field(min_length=1, max_length=100)
    precio:   float = Field(gt=0)
    stock:    int   = Field(ge=0, default=0)
    activo:   bool  = True

# FastAPI parsea el JSON del body, valida contra el modelo
# y devuelve 422 automáticamente si hay errores
@app.post("/productos")
def crear(producto: Producto):
    return producto
```

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional

app4 = FastAPI(title="API con Pydantic")

# Modelos Pydantic
class ProductoBase(BaseModel):
    nombre:    str   = Field(min_length=1, max_length=100, examples=["Teclado mecánico"])
    precio:    float = Field(gt=0, description="Precio en euros")
    stock:     int   = Field(ge=0, default=0)
    categoria: str   = Field(min_length=1)

class ProductoCreate(ProductoBase):
    pass  # mismo modelo para creación

class ProductoUpdate(BaseModel):
    nombre:    Optional[str]   = Field(None, min_length=1)
    precio:    Optional[float] = Field(None, gt=0)
    stock:     Optional[int]   = Field(None, ge=0)
    categoria: Optional[str]   = None

class ProductoResponse(ProductoBase):
    id: int

# Base de datos en memoria
db: dict[int, dict] = {}
contador = 0

@app4.get("/productos", response_model=list[ProductoResponse])
def listar():
    return list(db.values())

@app4.get("/productos/{prod_id}", response_model=ProductoResponse)
def obtener(prod_id: int):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Producto no encontrado")
    return db[prod_id]

@app4.post("/productos", response_model=ProductoResponse, status_code=201)
def crear(producto: ProductoCreate):
    global contador
    contador += 1
    nuevo = {"id": contador, **producto.model_dump()}
    db[contador] = nuevo
    return nuevo

@app4.put("/productos/{prod_id}", response_model=ProductoResponse)
def reemplazar(prod_id: int, producto: ProductoCreate):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Producto no encontrado")
    db[prod_id] = {"id": prod_id, **producto.model_dump()}
    return db[prod_id]

@app4.patch("/productos/{prod_id}", response_model=ProductoResponse)
def actualizar(prod_id: int, cambios: ProductoUpdate):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Producto no encontrado")
    datos = db[prod_id]
    for campo, valor in cambios.model_dump(exclude_none=True).items():
        datos[campo] = valor
    return datos

@app4.delete("/productos/{prod_id}", status_code=204)
def eliminar(prod_id: int):
    if prod_id not in db:
        raise HTTPException(status_code=404, detail="Producto no encontrado")
    del db[prod_id]


cfg4 = uvicorn.Config(app4, host="127.0.0.1", port=8004, log_level="warning")
srv4 = uvicorn.Server(cfg4)
threading.Thread(target=srv4.run, daemon=True).start()
time.sleep(1.2)

BASE4 = "http://127.0.0.1:8004"

# CRUD completo
print("=== POST (crear) ===")
p1 = requests.post(f"{BASE4}/productos",
                   json={"nombre": "Teclado mecánico", "precio": 85.0, "stock": 50, "categoria": "informatica"})
print(f"  {p1.status_code}: {p1.json()}")

p2 = requests.post(f"{BASE4}/productos",
                   json={"nombre": "Ratón inalámbrico", "precio": 45.0, "stock": 100, "categoria": "informatica"})

print("\n=== POST con datos inválidos (422) ===")
r = requests.post(f"{BASE4}/productos",
                  json={"nombre": "", "precio": -10, "categoria": "x"})  # nombre vacío, precio negativo
print(f"  {r.status_code}: {len(r.json()['detail'])} errores de validación")

print("\n=== GET lista ===")
print(requests.get(f"{BASE4}/productos").json())

print("\n=== PATCH (actualizar precio) ===")
r = requests.patch(f"{BASE4}/productos/1", json={"precio": 79.0})
print(f"  {r.status_code}: {r.json()}")

print("\n=== DELETE ===")
r = requests.delete(f"{BASE4}/productos/2")
print(f"  {r.status_code}")
print(f"  Quedan: {requests.get(f'{BASE4}/productos').json()}")

---
## 9. Práctica — API de tareas

---

### Ejercicio 1 — Modelo con validación

**Enunciado:** Define los modelos Pydantic para una API de gestión de tareas de equipo:

- `TareaCreate`: titulo (1-200 chars), descripcion (opcional), prioridad (`baja|normal|alta|urgente`), fecha_limite (string ISO opcional), asignado_a (str opcional)
- `TareaUpdate`: todos los campos opcionales
- `TareaResponse`: id (int), creada_en (str), completada (bool), + todos los de TareaCreate

Valida que los modelos funcionan: crea instancias válidas e inválidas.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
from enum import Enum
from datetime import datetime

class Prioridad(str, Enum):
    baja    = "baja"
    normal  = "normal"
    alta    = "alta"
    urgente = "urgente"

class TareaCreate(BaseModel):
    titulo:       str                   = Field(min_length=1, max_length=200)
    descripcion:  Optional[str]         = None
    prioridad:    Prioridad             = Prioridad.normal
    fecha_limite: Optional[str]         = None
    asignado_a:   Optional[str]         = None

class TareaUpdate(BaseModel):
    titulo:       Optional[str]         = Field(None, min_length=1, max_length=200)
    descripcion:  Optional[str]         = None
    prioridad:    Optional[Prioridad]   = None
    fecha_limite: Optional[str]         = None
    asignado_a:   Optional[str]         = None
    completada:   Optional[bool]        = None

class TareaResponse(TareaCreate):
    id:          int
    creada_en:   str
    completada:  bool = False

# Válida
t = TareaCreate(titulo="Revisar PR", prioridad="alta", asignado_a="Ana")
print("Válida:", t)

# Inválida: título vacío
try:
    TareaCreate(titulo="", prioridad="normal")
except ValidationError as e:
    print(f"Error validación: {e.error_count()} errores")

# Inválida: prioridad fuera de enum
try:
    TareaCreate(titulo="Test", prioridad="maxima")
except ValidationError as e:
    print(f"Error enum: {e.errors()[0]['msg']}")
```
</details>

---
### Ejercicio 2 — API completa de tareas

**Enunciado:** Implementa la API de tareas completa con todos los endpoints:

- `GET /tareas` — lista, con filtros opcionales: `?completada=true`, `?prioridad=alta`, `?asignado_a=Ana`
- `GET /tareas/{id}` — obtener una
- `POST /tareas` — crear
- `PATCH /tareas/{id}` — actualizar campos
- `POST /tareas/{id}/completar` — marcar como completada
- `DELETE /tareas/{id}` — eliminar
- `GET /tareas/estadisticas` — total, completadas, pendientes por prioridad

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum
import uvicorn, threading, time, requests


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field
from typing import Optional
from enum import Enum
import uvicorn, threading, time, requests
from datetime import datetime

class Prioridad(str, Enum):
    baja = "baja"; normal = "normal"; alta = "alta"; urgente = "urgente"

class TareaCreate(BaseModel):
    titulo:       str             = Field(min_length=1, max_length=200)
    descripcion:  Optional[str]   = None
    prioridad:    Prioridad       = Prioridad.normal
    asignado_a:   Optional[str]   = None

class TareaUpdate(BaseModel):
    titulo:       Optional[str]       = Field(None, min_length=1)
    descripcion:  Optional[str]       = None
    prioridad:    Optional[Prioridad] = None
    asignado_a:   Optional[str]       = None
    completada:   Optional[bool]      = None

class TareaResponse(TareaCreate):
    id: int; creada_en: str; completada: bool = False


app_tareas = FastAPI(title="API de Tareas")
bd: dict[int, dict] = {}
seq = 0

@app_tareas.get("/tareas", response_model=list[TareaResponse])
def listar(completada: Optional[bool] = None, prioridad: Optional[Prioridad] = None,
           asignado_a: Optional[str] = None):
    items = list(bd.values())
    if completada  is not None: items = [t for t in items if t['completada'] == completada]
    if prioridad:               items = [t for t in items if t['prioridad'] == prioridad]
    if asignado_a:              items = [t for t in items if t.get('asignado_a') == asignado_a]
    return items

@app_tareas.get("/tareas/estadisticas")
def estadisticas():
    todas = list(bd.values())
    return {
        "total":      len(todas),
        "completadas":sum(1 for t in todas if t['completada']),
        "pendientes": sum(1 for t in todas if not t['completada']),
        "por_prioridad": {p.value: sum(1 for t in todas if t['prioridad'] == p.value) for p in Prioridad}
    }

@app_tareas.get("/tareas/{id}", response_model=TareaResponse)
def obtener(id: int):
    if id not in bd: raise HTTPException(404, "Tarea no encontrada")
    return bd[id]

@app_tareas.post("/tareas", response_model=TareaResponse, status_code=201)
def crear(t: TareaCreate):
    global seq; seq += 1
    nueva = {"id": seq, "creada_en": datetime.now().isoformat(), "completada": False, **t.model_dump()}
    bd[seq] = nueva; return nueva

@app_tareas.patch("/tareas/{id}", response_model=TareaResponse)
def actualizar(id: int, cambios: TareaUpdate):
    if id not in bd: raise HTTPException(404, "Tarea no encontrada")
    for k, v in cambios.model_dump(exclude_none=True).items():
        bd[id][k] = v
    return bd[id]

@app_tareas.post("/tareas/{id}/completar", response_model=TareaResponse)
def completar(id: int):
    if id not in bd: raise HTTPException(404, "Tarea no encontrada")
    bd[id]['completada'] = True; return bd[id]

@app_tareas.delete("/tareas/{id}", status_code=204)
def eliminar(id: int):
    if id not in bd: raise HTTPException(404, "Tarea no encontrada")
    del bd[id]


cfg = uvicorn.Config(app_tareas, host="127.0.0.1", port=8005, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)

B = "http://127.0.0.1:8005"
requests.post(f"{B}/tareas", json={"titulo": "Revisar PR", "prioridad": "alta", "asignado_a": "Ana"})
requests.post(f"{B}/tareas", json={"titulo": "Daily standup", "prioridad": "normal"})
requests.post(f"{B}/tareas", json={"titulo": "Deploy a prod", "prioridad": "urgente", "asignado_a": "Ana"})
requests.post(f"{B}/tareas/{1}/completar")
print("Estadísticas:", requests.get(f"{B}/tareas/estadisticas").json())
print("Tareas de Ana:", [t['titulo'] for t in requests.get(f"{B}/tareas", params={"asignado_a": "Ana"}).json()])
```
</details>

---
### Ejercicio 3 — Errores personalizados

**Enunciado:** Añade a la API del ejercicio 2 las siguientes reglas de negocio con errores HTTP apropiados:

1. No se puede completar una tarea que ya está completada → 409 Conflict.
2. No se puede eliminar una tarea urgente → 403 Forbidden.
3. El título debe ser único entre las tareas activas → 409 Conflict.
4. Añade un endpoint `GET /tareas/{id}/historial` que por ahora devuelve un 501 Not Implemented.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
# Añadir las validaciones a los endpoints existentes:

# En crear:
activos = [t for t in bd.values() if not t['completada']]
if any(t['titulo'].lower() == t_create.titulo.lower() for t in activos):
    raise HTTPException(status_code=409, detail="Ya existe una tarea activa con ese título")

# En completar:
if bd[id]['completada']:
    raise HTTPException(status_code=409, detail="La tarea ya está completada")

# En eliminar:
if bd[id]['prioridad'] == 'urgente':
    raise HTTPException(status_code=403, detail="No se pueden eliminar tareas urgentes")

# Nuevo endpoint:
@app_tareas.get("/tareas/{id}/historial")
def historial(id: int):
    if id not in bd:
        raise HTTPException(status_code=404, detail="Tarea no encontrada")
    raise HTTPException(status_code=501, detail="Historial no implementado todavía")
```
</details>

---
### Ejercicio 4 — Sub-recursos

**Enunciado:** Amplía la API con comentarios en las tareas:

- `GET  /tareas/{id}/comentarios` — lista comentarios de una tarea
- `POST /tareas/{id}/comentarios` — añade un comentario
- Modelo `Comentario`: id, texto (1-500 chars), autor, creado_en
- Si se intenta comentar una tarea que no existe → 404

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
from pydantic import BaseModel, Field
from datetime import datetime

class ComentarioCreate(BaseModel):
    texto: str = Field(min_length=1, max_length=500)
    autor: str = Field(min_length=1)

class ComentarioResponse(ComentarioCreate):
    id: int
    tarea_id: int
    creado_en: str

comentarios: dict[int, list] = {}  # {tarea_id: [comentario, ...]}
seq_com = 0

@app_tareas.get("/tareas/{id}/comentarios", response_model=list[ComentarioResponse])
def listar_comentarios(id: int):
    if id not in bd:
        raise HTTPException(404, "Tarea no encontrada")
    return comentarios.get(id, [])

@app_tareas.post("/tareas/{id}/comentarios", response_model=ComentarioResponse, status_code=201)
def crear_comentario(id: int, comentario: ComentarioCreate):
    global seq_com
    if id not in bd:
        raise HTTPException(404, "Tarea no encontrada")
    seq_com += 1
    nuevo = {"id": seq_com, "tarea_id": id, "creado_en": datetime.now().isoformat(),
             **comentario.model_dump()}
    comentarios.setdefault(id, []).append(nuevo)
    return nuevo
```
</details>

---
### Ejercicio 5 (reto) — API de votaciones

**Enunciado:** Implementa desde cero una API de votaciones para decisiones de equipo:

- **Preguntas**: titulo, descripcion, opciones (list[str], mínimo 2), cerrada (bool)
- `GET /preguntas` — lista todas
- `POST /preguntas` — crea una pregunta
- `POST /preguntas/{id}/votar` — vota por una opción (`{"opcion": "Opción A", "votante": "Ana"}`)
  - Validar que la opción exista
  - Validar que la pregunta no esté cerrada
  - Un votante solo puede votar una vez por pregunta
- `POST /preguntas/{id}/cerrar` — cierra la votación
- `GET /preguntas/{id}/resultado` — muestra los votos y la opción ganadora

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import Optional
import uvicorn, threading, time, requests
from datetime import datetime

class PreguntaCreate(BaseModel):
    titulo:      str       = Field(min_length=1, max_length=200)
    descripcion: Optional[str] = None
    opciones:    list[str] = Field(min_length=2)

class VotoCreate(BaseModel):
    opcion:  str = Field(min_length=1)
    votante: str = Field(min_length=1)

app_votos = FastAPI(title="API Votaciones")
preguntas: dict[int, dict] = {}
votos: dict[int, list] = {}     # {pregunta_id: [{opcion, votante}]}
seq = 0

@app_votos.post("/preguntas", status_code=201)
def crear_pregunta(p: PreguntaCreate):
    global seq; seq += 1
    nueva = {"id": seq, "cerrada": False, "creada_en": datetime.now().isoformat(), **p.model_dump()}
    preguntas[seq] = nueva; votos[seq] = []
    return nueva

@app_votos.get("/preguntas")
def listar(): return list(preguntas.values())

@app_votos.post("/preguntas/{id}/votar")
def votar(id: int, voto: VotoCreate):
    if id not in preguntas: raise HTTPException(404, "Pregunta no encontrada")
    p = preguntas[id]
    if p['cerrada']:         raise HTTPException(409, "La votación está cerrada")
    if voto.opcion not in p['opciones']: raise HTTPException(400, "Opción no válida")
    ya_voto = any(v['votante'] == voto.votante for v in votos[id])
    if ya_voto: raise HTTPException(409, "Este votante ya ha votado")
    votos[id].append(voto.model_dump())
    return {"mensaje": "Voto registrado", "votos_totales": len(votos[id])}

@app_votos.post("/preguntas/{id}/cerrar")
def cerrar(id: int):
    if id not in preguntas: raise HTTPException(404, "Pregunta no encontrada")
    preguntas[id]['cerrada'] = True
    return {"mensaje": "Votación cerrada"}

@app_votos.get("/preguntas/{id}/resultado")
def resultado(id: int):
    if id not in preguntas: raise HTTPException(404, "Pregunta no encontrada")
    p = preguntas[id]
    conteo = {op: sum(1 for v in votos[id] if v['opcion'] == op) for op in p['opciones']}
    ganadora = max(conteo, key=conteo.get) if votos[id] else None
    return {"pregunta": p['titulo'], "votos": conteo, "ganadora": ganadora, "total": len(votos[id])}


cfg = uvicorn.Config(app_votos, host="127.0.0.1", port=8006, log_level="warning")
threading.Thread(target=uvicorn.Server(cfg).run, daemon=True).start()
time.sleep(1.2)

B = "http://127.0.0.1:8006"
p = requests.post(f"{B}/preguntas", json={"titulo": "¿Dónde hacemos el team building?",
    "opciones": ["Madrid", "Barcelona", "Online"]}).json()
print("Pregunta:", p['titulo'])

for votante, opcion in [("Ana","Madrid"),("Carlos","Barcelona"),("Lucía","Madrid"),("Pedro","Online")]:
    r = requests.post(f"{B}/preguntas/1/votar", json={"opcion": opcion, "votante": votante})
    print(f"  {votante} → {opcion}: {r.status_code}")

# Doble voto
r = requests.post(f"{B}/preguntas/1/votar", json={"opcion": "Online", "votante": "Ana"})
print(f"  Ana vota de nuevo: {r.status_code} — {r.json()['detail']}")

print("\nResultado:", requests.get(f"{B}/preguntas/1/resultado").json())
```
</details>

---
## Resumen de la clase

```python
from fastapi import FastAPI, HTTPException, Path, Query
from pydantic import BaseModel, Field

app = FastAPI()

class Item(BaseModel):
    nombre: str = Field(min_length=1)
    precio: float = Field(gt=0)

# Path param con validación
@app.get("/items/{id}")
def get(id: int = Path(ge=1)):
    ...

# Query params con defaults
@app.get("/items")
def listar(skip: int = 0, limit: int = Query(10, le=100)):
    ...

# Body JSON → Pydantic valida automáticamente
@app.post("/items", status_code=201)
def crear(item: Item):
    ...

# Error HTTP
raise HTTPException(status_code=404, detail="No encontrado")

# Lanzar:
# uvicorn main:app --reload
# Docs: http://localhost:8000/docs
```

## ¿Qué viene en la próxima clase?

En la **Clase 4** conectamos FastAPI con una base de datos real usando SQLAlchemy: persistencia de datos, modelos ORM y operaciones CRUD que sobreviven al reinicio del servidor.